# M2 Deep Learning Coursework: Normalizing Flows

This notebook is a scaffold for the final coursework implementation.
All major required sections are present as TODO placeholders.


## 0) Setup and Reproducibility


In [1]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import torch
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from visualization_utils import (
    plot_correctness_figure,
    plot_data_splits,
    plot_logprob_landscape,
    plot_roundtrip_overlay,
)


In [2]:
# TODO: keep this seed and deterministic setup in the final implementation.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

# CPU-only coursework assumption
DEVICE = torch.device('cpu')


## 1) Data Loading

Load the provided train/validation/test CSV files into tensors.
The `class` column is ignored for flow training and used only for diagnostics/plots.


In [3]:
# Load CSV files; class labels are retained only for diagnostics (not training).
def load_moons_csv(path: Path):
    arr = np.genfromtxt(path, delimiter=',', names=True, dtype=np.float64)
    x = np.stack([arr['x1'], arr['x2']], axis=1).astype(np.float64)
    cls = arr['class'].astype(np.int64)
    return torch.from_numpy(x).to(DEVICE), torch.from_numpy(cls)

train_x, train_class = load_moons_csv(Path('data/moons_train.csv'))
val_x, val_class = load_moons_csv(Path('data/moons_val.csv'))
test_x, test_class = load_moons_csv(Path('data/moons_test.csv'))

for name, split_x in [('train', train_x), ('val', val_x), ('test', test_x)]:
    if split_x.ndim != 2 or split_x.shape[1] != 2:
        raise ValueError(f'{name} split must have shape [N, 2], got {tuple(split_x.shape)}')

Path('figs').mkdir(parents=True, exist_ok=True)
plot_data_splits(
    train_x=train_x,
    train_class=train_class,
    val_x=val_x,
    val_class=val_class,
    test_x=test_x,
    test_class=test_class,
    out_path='figs/Figure1a_data_overview.pdf',
)

display(Markdown('**Figure 1a.** Train/val/test splits share the same two-moons geometry; class labels are used only for diagnostics.'))
display(Image(filename='figs/Figure1a_data_overview.png', width=920))

print(f'train/val/test sizes: {len(train_x)}/{len(val_x)}/{len(test_x)}')
print('Saved figs/Figure1a_data_overview.pdf')


train/val/test sizes: 800/100/100


## 2) Model Definition (2D Affine Coupling Flow)

Implement the required affine coupling layer and stack alternating masks to form the full flow.
This section enforces coursework constraints (`hidden <= 128`, `n_layers <= 8`) and defines batch `log p(x)`.


In [4]:
import math
import torch.nn as nn

class AffineCoupling2D(nn.Module):
    """Single 2D affine coupling layer with configurable binary mask."""

    def __init__(self, *, dim: int, hidden: int, mask: torch.Tensor):
        super().__init__()
        if dim != 2:
            raise ValueError('This coursework implementation expects dim=2.')
        if hidden > 128:
            raise ValueError('Coursework constraint violated: hidden must be <= 128.')
        if mask.shape != (dim,):
            raise ValueError(f'mask must have shape ({dim},), got {tuple(mask.shape)}')

        self.dim = dim
        self.hidden = hidden
        self.register_buffer('mask', mask.to(dtype=torch.float64))

        # Required MLP: Linear(D->H) -> ReLU -> Linear(H->2D), then split into s/t.
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2 * dim),
        )

    def _st(self, x: torch.Tensor):
        x_masked = x * self.mask
        st = self.net(x_masked)
        s, t = torch.chunk(st, 2, dim=-1)
        s = torch.tanh(s)

        transform_mask = 1.0 - self.mask
        s = s * transform_mask
        t = t * transform_mask
        return s, t

    def forward(self, z: torch.Tensor):
        """Forward map f: base -> data. Returns transformed output and log|det J_f|."""
        s, t = self._st(z)
        transform_mask = 1.0 - self.mask
        x = z * self.mask + transform_mask * (z * torch.exp(s) + t)
        logdet = s.sum(dim=-1)
        return x, logdet

    def inverse(self, x: torch.Tensor):
        """Inverse map f^{-1}: data -> base. Returns output and log|det J_{f^{-1}}|."""
        s, t = self._st(x)
        transform_mask = 1.0 - self.mask
        z = x * self.mask + transform_mask * ((x - t) * torch.exp(-s))
        logdet = -s.sum(dim=-1)
        return z, logdet


class CouplingFlow2D(nn.Module):
    """Stack of affine coupling layers with alternating masks for D=2."""

    def __init__(self, *, n_layers: int, hidden: int, dim: int = 2):
        super().__init__()
        if n_layers < 1 or n_layers > 8:
            raise ValueError('Coursework constraint violated: n_layers must be in [1, 8].')
        if hidden < 1 or hidden > 128:
            raise ValueError('Coursework constraint violated: hidden must be in [1, 128].')

        self.dim = dim
        self.n_layers = n_layers
        self.hidden = hidden

        layers = []
        for i in range(n_layers):
            mask = torch.tensor([1.0, 0.0]) if i % 2 == 0 else torch.tensor([0.0, 1.0])
            layers.append(AffineCoupling2D(dim=dim, hidden=hidden, mask=mask))
        self.layers = nn.ModuleList(layers)

    def forward(self, z: torch.Tensor):
        x = z
        total_logdet = torch.zeros(z.shape[0], dtype=z.dtype, device=z.device)
        for layer in self.layers:
            x, logdet = layer.forward(x)
            total_logdet = total_logdet + logdet
        return x, total_logdet

    def inverse(self, x: torch.Tensor):
        z = x
        total_logdet = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
        for layer in reversed(self.layers):
            z, logdet = layer.inverse(z)
            total_logdet = total_logdet + logdet
        return z, total_logdet

    @staticmethod
    def base_log_prob(z: torch.Tensor):
        log_2pi = math.log(2.0 * math.pi)
        return -0.5 * (z.pow(2) + log_2pi).sum(dim=-1)

    def log_prob(self, x: torch.Tensor):
        z, inv_logdet = self.inverse(x)
        return self.base_log_prob(z) + inv_logdet


# Compliant model size: hidden <= 128, n_layers <= 8.
FLOW_CONFIG = {'dim': 2, 'hidden': 64, 'n_layers': 6}
flow = CouplingFlow2D(
    dim=FLOW_CONFIG['dim'],
    hidden=FLOW_CONFIG['hidden'],
    n_layers=FLOW_CONFIG['n_layers'],
).to(DEVICE).double()
flow.eval()

plot_logprob_landscape(
    flow=flow,
    reference_x=train_x,
    out_path='figs/Figure1b_untrained_logprob.pdf',
)
display(Markdown('**Figure 1b.** Before training, the flow induces a smooth density field and does not yet capture the moons manifold.'))
display(Image(filename='figs/Figure1b_untrained_logprob.png', width=760))
print('Saved figs/Figure1b_untrained_logprob.pdf')


CouplingFlow2D(
  (layers): ModuleList(
    (0-5): 6 x AffineCoupling2D(
      (net): Sequential(
        (0): Linear(in_features=2, out_features=64, bias=True)
        (1): ReLU()
        (2): Linear(in_features=64, out_features=4, bias=True)
      )
    )
  )
)

## 3) Correctness Checks

Run the two required checks: full-train-set invertibility and finite-difference inverse log-determinant validation (`epsilon=1e-4`).
Export metrics to `results.json` and save `figs/Figure1c.pdf`.


In [5]:
# Correctness check 1: invertibility over all training points.
with torch.no_grad():
    z_train, inv_logdet_train = flow.inverse(train_x)
    x_hat, fwd_logdet_train = flow.forward(z_train)

reconstruction_abs = (x_hat - train_x).abs()
invertibility_max_abs_error = float(reconstruction_abs.max().item())

# Correctness check 2: analytic vs finite-difference inverse log-det on first training example.
epsilon = 1e-4
x0 = train_x[:1].clone()
with torch.no_grad():
    _, analytic_inv_logdet = flow.inverse(x0)
analytic_inv_logdet_value = float(analytic_inv_logdet.item())

jacobian_cols = []
for d in range(x0.shape[1]):
    delta = torch.zeros_like(x0)
    delta[0, d] = epsilon
    with torch.no_grad():
        z_plus, _ = flow.inverse(x0 + delta)
        z_minus, _ = flow.inverse(x0 - delta)
    col = ((z_plus - z_minus) / (2.0 * epsilon)).squeeze(0)
    jacobian_cols.append(col)

jacobian = torch.stack(jacobian_cols, dim=1)
finite_diff_logabsdet = float(torch.log(torch.abs(torch.det(jacobian))).item())
logdet_finite_diff_abs_error = float(abs(finite_diff_logabsdet - analytic_inv_logdet_value))

plot_correctness_figure(
    reconstruction_abs=reconstruction_abs,
    invertibility_max_abs_error=invertibility_max_abs_error,
    analytic_inv_logdet_value=analytic_inv_logdet_value,
    finite_diff_logabsdet=finite_diff_logabsdet,
    logdet_finite_diff_abs_error=logdet_finite_diff_abs_error,
    out_path='figs/Figure1c.pdf',
)

plot_roundtrip_overlay(
    x=train_x,
    x_hat=x_hat,
    out_path='figs/Figure1d_roundtrip_overlay.pdf',
)

display(Markdown('**Figure 1c.** Both required checks pass: reconstruction error is near machine precision and analytic vs finite-difference log-determinants agree.'))
display(Image(filename='figs/Figure1c.png', width=920))
display(Markdown('**Figure 1d.** Roundtrip overlay confirms `x` and `x_hat` coincide up to numerical precision.'))
display(Image(filename='figs/Figure1d_roundtrip_overlay.png', width=920))

# Save required correctness outputs for this step.
results = {
    'correctness': {
        'invertibility_max_abs_error': float(invertibility_max_abs_error),
        'logdet_finite_diff_abs_error': float(logdet_finite_diff_abs_error),
    }
}

with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print('Saved figs/Figure1c.pdf')
print('Saved figs/Figure1d_roundtrip_overlay.pdf')


{
  "correctness": {
    "invertibility_max_abs_error": 9.992007221626409e-16,
    "logdet_finite_diff_abs_error": 1.4130335790341064e-11
  }
}
Saved figs/Figure1c.pdf


## 4) Training on Tiny Subset (First 128 Rows)


In [6]:
# TODO: Train on the first 128 training rows only.
# TODO: Record tiny-set loss curve and save figs/Figure2a.pdf.
# TODO: Store tinyset_final_nll for results.json.


## 5) Full Training


In [7]:
# TODO: Train on full training set with a learning-rate schedule (<=10,000 optimizer steps).
# TODO: Track train/validation curves and save figs/Figure2c.pdf.
# TODO: Save checkpoint to checkpoints/flow_full.pt with keys: state_dict, config, seed.
# TODO: Save logs/training_curves.json with keys: tiny_loss, full_loss, full_val_loss.


## 6) Flow Surgery (One-Parameter Family)


In [8]:
# TODO: Implement g_alpha and g_alpha^{-1}.
# TODO: Load checkpoints/flow_full.pt and define f_alpha(z)=g_alpha(f0(z)) without retraining.
# TODO: Generate panel for alpha in {-2, -1, 0, 1, 2} and save figs/Figure3b.pdf.


## 7) FLOP Counting


In [9]:
def count_flops(*, dim: int, n_layers: int, hidden: int, batch_size: int) -> int:
    """TODO: Implement coursework-specified analytical FLOP counting."""
    raise NotImplementedError('TODO: implement FLOP counting for log p(x) evaluation.')


## 8) Artifact Export


In [10]:
# TODO: Build results dictionary with required keys and write results.json.
# TODO: Ensure all required output files are generated by running this notebook top-to-bottom.
